# 08 — Design Space: δ_M vs x/d

This notebook maps the moment-redistribution factor $\delta_M$ across the
EC2-relevant design space for a two-span continuous RC beam.

**Primary parameter:** reinforcement area $A_s$ — swept directly over a physically
meaningful range.  For each $A_s$ the actual $x/d$ is extracted from the M-κ strain
state at the peak hogging moment (`cba.xd_hog_u`), and the EC2 rectangular
stress-block formula $x/d_\mathrm{EC2} = \omega/0.8$ is computed for the same $A_s$
as a comparison.  $\delta_M$ is obtained via `cba.delta_M_u`.

**Key finding:** for the symmetric doubly-reinforced section used here, the compression
steel (bottom bars, in compression at the interior support) nearly cancels the tension
steel in equilibrium, so the concrete compression zone stays very shallow — actual
$x/d$ saturates near $0.10$–$0.17$ regardless of $A_s$, while the EC2 singly-reinforced
formula grows linearly to $\sim 0.6$.  The section is far more ductile than EC2 assumes,
which is why $\delta_M$ lies well below the EC2 lower bound.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scite.beam.bending.beam_deflection import BeamDeflectionAnalysis
from scite.beam.continuous import ContinuousBeamAnalysis, CompatibilityNoRoot
from scite.cs_design.cross_section import CrossSection
from scite.cs_design.shapes import RectangularShape
from scite.cs_design.reinforcement import ReinforcementLayer, ReinforcementLayout
from scite.matmod.ec2_parabola_rectangle import EC2ParabolaRectangle
from scite.matmod.steel_reinforcement import SteelReinforcement
from scite.mkappa.mkappa import MKappaAnalysis

## Reference section — fixed parameters

All sweeps use the same rectangular cross-section with C30/37 concrete and B500B
reinforcement steel.  Section width $b = 300\,\text{mm}$, height $h = 600\,\text{mm}$,
cover to bar centroid $c = 50\,\text{mm}$ (top and bottom), giving effective depth
$d = 550\,\text{mm}$.  The reference left span is $L_a = 6\,\text{m}$.

In [ ]:
# ── Materials ────────────────────────────────────────────────────────────────
f_ck     = 30.0     # N/mm²  — characteristic compressive strength (C30/37)
alpha_cc = 0.85     #          EC2 §3.1.6
gamma_c  = 1.5      #          EC2 §2.4.2.4
f_cd     = alpha_cc * f_ck / gamma_c   # = 17.0 N/mm²

f_yk     = 500.0    # N/mm²  — characteristic yield strength (B500B)
gamma_s  = 1.15     #          EC2 §2.4.2.4
E_s      = 200_000.0  # N/mm²
f_yd     = f_yk / gamma_s              # ≈ 435 N/mm²

# ── Section geometry ─────────────────────────────────────────────────────────
b  = 300.0   # mm — width
h  = 600.0   # mm — total height
c  = 50.0    # mm — cover to bar centroid (top AND bottom)
d  = h - c   # mm — effective depth (= 550 mm)

# ── Reference span ───────────────────────────────────────────────────────────
La = 6_000.0  # mm — left span (right span = S·La)

# ── Shared material objects ───────────────────────────────────────────────────
concrete = EC2ParabolaRectangle(f_ck=f_ck, alpha_cc=alpha_cc, gamma_c=gamma_c)
steel    = SteelReinforcement(f_yk=f_yk, E_s=E_s, gamma_s=gamma_s)

print(f"f_cd = {f_cd:.1f} N/mm²,  f_yd = {f_yd:.1f} N/mm²,  d = {d:.0f} mm")

## Two-step approach: x/d sweep then δ_M sweep

The two quantities are determined independently:

**Step 1 — x/d from M-κ only (fast).**
For each $A_s$, build the cross-section and run `MKappaAnalysis.solve()` with a
coarse grid (`n_kappa=60`).  Read `mk.xd_neg` — the actual neutral-axis depth ratio
at the hogging peak.  No `BeamDeflectionAnalysis`, no rotation integration, no CBA.

The EC2 comparison value $x/d_\mathrm{EC2} = \omega/0.8$ is computed analytically:

$$\omega = \frac{A_s}{b\,d}\,\frac{f_{yd}}{f_{cd}}, \qquad
\frac{x}{d}\bigg|_\mathrm{EC2} = \frac{\omega}{0.8}$$

(singly-reinforced approximation — ignores the compression steel at the bottom.)

**Step 2 — δ_M from CBA (slower).**
Build the full `ContinuousBeamAnalysis` with `n_kappa=150` and call `cba.delta_M_u`.
This is a single compatibility root-find at $\kappa_R$ — not a full κ-sweep.

**Note on the EC2 formula:** EC2 §5.5(4) eq. (5.10a) ties $\delta_\min$ to the hogging
section's $x/d$ only.  It does not cut off redistribution when the sagging section
governs failure — it assumes the sagging section was separately designed with adequate
capacity.  In our symmetric section ($M_{R,\text{hog}} = M_{R,\text{sag}}$) the sagging
zone fails prematurely for low $A_s$; this will be addressed in a later step by using
a non-uniform reinforcement profile.

In [ ]:
def _build_cs(A_s: float) -> CrossSection:
    """Symmetric doubly-reinforced section with equal top/bottom steel."""
    top = ReinforcementLayer(A_s=A_s, z=c,     material=steel)
    bot = ReinforcementLayer(A_s=A_s, z=h - c, material=steel)
    return CrossSection(
        shape=RectangularShape(b=b, h=h),
        concrete=concrete,
        reinforcement=ReinforcementLayout(layers=[top, bot]),
    )

def _make_mk(cs: CrossSection) -> MKappaAnalysis:
    return MKappaAnalysis(cs=cs, n_kappa=60, solve_mode='negative')

def make_cba_from_As(A_s: float, S: float = 1.0) -> ContinuousBeamAnalysis:
    """Build a CBA with equal top/bottom steel A_s.

    When S=1 the same BDA instance is shared between both spans (saves one
    redundant M-κ solve that would otherwise be identical).
    """
    cs = _build_cs(A_s)
    bda_left = BeamDeflectionAnalysis(cs=cs, L=La, load_type='dist',
                                      n_kappa=150, solve_mode='both')
    if S == 1.0:
        bda_right = bda_left
    else:
        bda_right = BeamDeflectionAnalysis(cs=cs, L=S * La, load_type='dist',
                                           n_kappa=150, solve_mode='both')
    return ContinuousBeamAnalysis(span_left=bda_left, span_right=bda_right, q=1.0)

## Verification — single design point

For $A_s = 800\,\text{mm}^2$, compare the actual $x/d$ from the M-κ model with the
EC2 rectangular stress-block formula, then compute $\delta_M$.

In [ ]:
A_s_check = 800.0

# x/d — MKappaAnalysis only
mk_check  = _make_mk(_build_cs(A_s_check))
mk_check.solve()
xd_actual = mk_check.xd_neg
xd_ec2    = (A_s_check / (b * d)) * f_yd / f_cd / 0.8

print(f"─── Section ───────────────────────────────────────────────")
print(f"  A_s           = {A_s_check:.0f} mm²  (equal top + bottom)")
print()
print(f"─── x/d comparison ────────────────────────────────────────")
print(f"  x/d  M-κ model    = {xd_actual:.4f}  (from actual strain state at κ_R)")
print(f"  x/d  EC2 formula  = {xd_ec2:.4f}  (singly-reinforced approximation)")
print(f"  Δ(x/d) = model − EC2 = {xd_actual - xd_ec2:+.4f}  ({(xd_actual/xd_ec2 - 1)*100:+.2f} %)")
print()

# δ_M — ContinuousBeamAnalysis
cba_check = make_cba_from_As(A_s_check)
delta_M   = cba_check.delta_M_u
print(f"─── Redistribution ─────────────────────────────────────────")
print(f"  δ_M             = {delta_M:.4f}")
print(f"  EC2 lower bound = {0.64 + 0.8 * xd_actual:.4f}  (at xd_model)")

## EC2 §5.5 redistribution limit (eq. 5.10a)

Original German text (*DIN EN 1992-1-1 mit Nationalem Anhang*, §5.5 (4), eq. 5.10a):

> *für $f_{ck} \leq 50\,\text{N/mm}^2$:*
> $$\delta \geq 0.64 + 0.8\,\frac{x_u}{d}$$
> *mit dem oberen Grenzwert $\delta \leq 0.80$ für B500B.*

Here $\delta$ is the ratio of the redistributed support moment to the elastic support
moment, and $x_u/d$ is the relative neutral-axis depth at the ULS after redistribution.

In the simulation:
- $x/d$ comes from `cba.xd_hog_u` — the strain state at the peak of the hogging
  M-κ curve (`kappa_R_hog`).
- $q_{u,\text{hog}}$ is found by a single compatibility root-find at that same
  $\kappa_R$ via `solve_q_for_kappa_hog` — no full κ-sweep is needed.
- $\delta_M = |M_\text{nl,hog}| / |M_\text{el,hog}(q_u)|$ is then formed directly.

## Step 1: x/d sweep — M-κ model only (equal spans, S = 1)

For each $A_s$ in the range 200–3000 mm², build the cross-section and run
`MKappaAnalysis.solve()` with `n_kappa=60` (coarse — sufficient for peak detection).
Read `mk.xd_neg` for the actual hogging x/d; compute `xd_ec2 = ω/0.8` analytically
for the same $A_s$.  No BDA, no CBA, no rotation integration.

**Key observation:** for the symmetric doubly-reinforced section the compression steel
(bottom bars) nearly cancels the tension steel in equilibrium, keeping the concrete
compression zone very shallow — $x/d_\mathrm{model}$ saturates near 0.10–0.17 regardless
of $A_s$, while $x/d_\mathrm{EC2}$ (singly-reinforced formula) grows linearly to ~0.58.

In [ ]:
# ── Step 1: x/d sweep — MKappaAnalysis only ──────────────────────────────────
A_s_range    = np.linspace(200, 3000, 12)
xd_actual_S1 = []
xd_ec2_S1    = []

mk = _make_mk(_build_cs(30000))

In [ ]:
# ── Step 1: x/d sweep — MKappaAnalysis only ──────────────────────────────────
A_s_range    = np.linspace(200, 3000, 12)
xd_actual_S1 = []
xd_ec2_S1    = []

for A_s in A_s_range:
    mk = _make_mk(_build_cs(A_s))
    mk.solve()
    xda = mk.xd_neg
    xde = (A_s / (b * d)) * f_yd / f_cd / 0.8
    xd_actual_S1.append(xda)
    xd_ec2_S1.append(xde)
    print(f"A_s={A_s:7.1f} mm²  xd_model={xda:.3f}  xd_EC2={xde:.3f}  "
          f"ratio={xda/xde:.2f}")

xd_actual_S1 = np.array(xd_actual_S1)
xd_ec2_S1    = np.array(xd_ec2_S1)

## Step 2: δ_M sweep — ContinuousBeamAnalysis (equal spans, S = 1)

Run the CBA for the same $A_s$ range.  `cba.delta_M_u` uses a single compatibility
root-find at $\kappa_R$ — no full κ-sweep.  The x/d values here come from the fast
MKappaAnalysis sweep above; they are not re-extracted from the CBA.

In [ ]:
# ── Step 2: δ_M sweep — ContinuousBeamAnalysis ───────────────────────────────
delta_M_S1 = []

for A_s in A_s_range:
    cba = make_cba_from_As(A_s, S=1.0)
    dM  = cba.delta_M_u
    delta_M_S1.append(dM)
    print(f"A_s={A_s:7.1f} mm²  δ_M={dM:.4f}")

delta_M_S1 = np.array(delta_M_S1)

In [ ]:
xd_plot   = np.linspace(0.05, 0.50, 200)
delta_ec2 = 0.64 + 0.8 * xd_plot

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(xd_actual_S1, delta_M_S1, 'o-', color='steelblue', lw=2,
        label=r'$\delta_M$ simulation (S = 1, x/d from M-κ model)')
ax.plot(xd_plot, delta_ec2, '--', color='#cc2222', lw=1.5,
        label=r'EC2 §5.5 eq. (5.10a): $\delta_{\min} = 0.64 + 0.8\,x_u/d$')

ax.axhline(1.0,  color='gray',    lw=0.8, ls=':')
ax.axhline(0.70, color='#888888', lw=0.8, ls=':',
           label=r'$\delta = 0.70$ (B500B lower bound)')

ax.set_xlabel(r'$x/d$ — actual neutral-axis depth from M-κ model')
ax.set_ylabel(r'$\delta_M$ — moment redistribution factor')
ax.set_title('Design space: $\\delta_M$ vs $x/d$\n'
             'C30/37, B500B, $S = 1$, $L_a = 6\\,000\\,\\mathrm{mm}$')
ax.legend(fontsize=9)
ax.set_xlim(0.05, 0.50)
ax.set_ylim(0.40, 1.10)
plt.tight_layout()

In [ ]:
# ── x/d comparison: M-κ model vs EC2 rectangular stress-block formula ────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: xd_EC2 vs xd_model (points fall far below identity line for large A_s)
ax = axes[0]
all_vals = list(xd_actual_S1) + list(xd_ec2_S1)
lims = [min(all_vals) * 0.90, max(all_vals) * 1.08]
ax.plot(lims, lims, '--', color='gray', lw=1.2, label='EC2 = model')
ax.plot(xd_ec2_S1, xd_actual_S1, 'o', color='steelblue', ms=7,
        label='design points')
ax.set_xlabel(r'$x/d$ — EC2 rectangular stress block (singly-reinforced)')
ax.set_ylabel(r'$x/d$ — M-κ model (doubly-reinforced, actual)')
ax.set_title(r'EC2 formula vs M-κ model: $x/d$ at peak hogging moment')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, ls='--')
ax.set_aspect('equal')

# Right: relative difference (model − EC2) / EC2 in percent
ax2 = axes[1]
rel_diff = (xd_actual_S1 - xd_ec2_S1) / xd_ec2_S1 * 100.0
ax2.plot(xd_ec2_S1, rel_diff, 'o-', color='#d62728', lw=1.8)
ax2.axhline(0.0, color='gray', lw=0.8, ls='--')
ax2.set_xlabel(r'$x/d$ — EC2 rectangular stress block')
ax2.set_ylabel(r'$(x/d_\mathrm{model} - x/d_\mathrm{EC2})\;/\;x/d_\mathrm{EC2}$  [%]')
ax2.set_title('Relative deviation: model vs EC2 stress-block formula')
ax2.grid(True, alpha=0.3, ls='--')

plt.tight_layout()
plt.show()

print("A_s [mm²]   xd_model   xd_EC2   deviation")
for A_s, xda, xde in zip(A_s_range, xd_actual_S1, xd_ec2_S1):
    print(f"  {A_s:7.1f}    {xda:.3f}      {xde:.3f}    {(xda/xde-1)*100:+.1f}%")

### Reading the plot

- **Blue curve** (simulation): redistribution factor $\delta_M$ from `cba.delta_M_u` —
  a single compatibility root-find at the hogging M-κ peak, not a full κ-sweep.
  A value below 1 means the beam carries load beyond the elastic hogging-moment limit
  by redistributing surplus moment to the sagging zone.
- **Red dashed line** (EC2): the *minimum* $\delta$ the designer is required to assume
  in a linear-elastic-with-redistribution design.  If the simulation curve lies **above**
  the EC2 line at the actual $x/d$, the section achieves less redistribution than EC2
  would allow — here because $M_{R,\text{hog}} = M_{R,\text{sag}}$ (symmetric section)
  limits the moment transfer before the sagging hinge forms.

The simulation directly verifies (or challenges) the empirical EC2 formula across the
full design range.

## Span ratio S as secondary parameter

For asymmetric spans ($S = L_b/L_a \neq 1$) the elastic hogging moment shifts and the
sagging capacity of the shorter (or longer) span may govern failure before the hogging
plastic hinge is fully mobilised.  This sweep identifies where the decoupling
($\delta_M$ independent of $L/d$) breaks down.

In [ ]:
S_range    = np.array([0.6, 0.7, 0.8, 0.9, 1.0])
A_s_range2 = np.linspace(200, 3000, 8)
DM_grid    = np.zeros((len(S_range), len(A_s_range2)))
xd_grid    = np.zeros_like(DM_grid)

for i, S in enumerate(S_range):
    for j, A_s in enumerate(A_s_range2):
        try:
            cba = make_cba_from_As(A_s, S=S)
            DM_grid[i, j] = cba.delta_M_u
            xd_grid[i, j] = cba.xd_hog_u
        except CompatibilityNoRoot:
            DM_grid[i, j] = np.nan
            xd_grid[i, j] = np.nan
    print(f"S = {S:.1f} done")

In [ ]:
colors = plt.cm.Blues(np.linspace(0.4, 1.0, len(S_range)))

fig, ax = plt.subplots(figsize=(8, 5))

for i, S in enumerate(S_range):
    mask = ~np.isnan(DM_grid[i])
    ax.plot(xd_grid[i][mask], DM_grid[i, mask], 'o-',
            color=colors[i], lw=1.8, label=f'$S = {S:.1f}$')

ax.plot(xd_plot, delta_ec2, '--', color='#cc2222', lw=1.5,
        label=r'EC2 eq. (5.10a)')
ax.axhline(1.0,  color='gray',    lw=0.7, ls=':')
ax.axhline(0.70, color='#888888', lw=0.7, ls=':')

ax.set_xlabel(r'$x/d$ — actual neutral-axis depth from M-κ model')
ax.set_ylabel(r'$\delta_M$')
ax.set_title(r'$\delta_M(x/d,\, S)$ — span ratio $S = L_b/L_a$ as secondary parameter')
ax.legend(fontsize=9, ncol=2)
ax.set_xlim(0.05, 0.50)
ax.set_ylim(0.40, 1.10)
plt.tight_layout()